# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

0. Импорты

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

1. Загрузите датасет и выведите на экран первые несколько строк

In [2]:
df = pd.read_csv('auto_dataset.csv')
df.head()

,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [3]:
cat_cols = ['brand', 'model', 'vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']
num_cols = ['powerPS', 'kilometer', 'autoAgeMonths']
target_col = 'price'

X = df[cat_cols + num_cols]
y = df[target_col].values.astype(np.float64)

3. Разбейте датасет на train val test в отношении 8:1:1

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer([
    ('cat', ohe, cat_cols),
    ('num', StandardScaler(), num_cols)
])

X_train = preprocessor.fit_transform(X_train).astype(np.float64)
X_val   = preprocessor.transform(X_val).astype(np.float64)
X_test  = preprocessor.transform(X_test).astype(np.float64)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_val_s   = y_scaler.transform(y_val.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

def add_bias(X):
    return np.hstack([X, np.ones((X.shape[0], 1))])

X_train_b = add_bias(X_train)
X_val_b   = add_bias(X_val)
X_test_b  = add_bias(X_test)

print('Train:', X_train_b.shape, '| Val:', X_val_b.shape, '| Test:', X_test_b.shape)

Train: (800, 195) | Val: (100, 195) | Test: (100, 195)


### Объявим функции

In [ ]:
MAX_ITER   = 10000
EVAL_EVERY = 50
PATIENCE   = 15
BATCH_SIZE = 128
S0 = 1.0
P  = 0.5


def mse_loss(X, y, w):
    """Q(w) = (1/l) * ||Xw - y||^2."""
    with np.errstate(over='ignore', invalid='ignore'):
        r = X @ w - y
        val = np.mean(r ** 2)
    return val if np.isfinite(val) else np.inf


def full_grad(X, y, w):
    """Градиент MSE: (2/l) * X^T (Xw - y)."""
    n = X.shape[0]
    with np.errstate(over='ignore', invalid='ignore'):
        g = 2.0 / n * X.T @ (X @ w - y)
    return g


def get_lr(lr, schedule, k):
    if schedule == 'constant':
        return lr
    elif schedule == 'time_decay':
        return lr * (S0 / (S0 + k)) ** P
    else:
        raise ValueError(schedule)


def final_metrics(w):
    pred_train = y_scaler.inverse_transform((X_train_b @ w).reshape(-1, 1)).ravel()
    pred_val   = y_scaler.inverse_transform((X_val_b   @ w).reshape(-1, 1)).ravel()
    pred_test  = y_scaler.inverse_transform((X_test_b  @ w).reshape(-1, 1)).ravel()
    return {
        'Loss_train': mean_squared_error(y_train, pred_train),
        'Loss_val':   mean_squared_error(y_val,   pred_val),
        'Loss_test':  mean_squared_error(y_test,  pred_test),
        'R2_train':   r2_score(y_train, pred_train),
        'R2_val':     r2_score(y_val,   pred_val),
        'R2_test':    r2_score(y_test,  pred_test),
    }

#### Vanilla Gradient Descent

In [83]:
def train_vgd(X, y, X_val, y_val, lr, schedule):
    n, d = X.shape
    w = np.zeros(d)
    best_val, best_w, best_iter, wait = np.inf, w.copy(), 0, 0

    for k in range(MAX_ITER):
        eta = get_lr(lr, schedule, k)
        grad = full_grad(X, y, w)
        
        with np.errstate(over='ignore', invalid='ignore'):
            w = w - eta * grad

        if (k + 1) % EVAL_EVERY == 0:
            vl = mse_loss(X_val, y_val, w)
            if vl < best_val:
                best_val, best_w, best_iter, wait = vl, w.copy(), k + 1, 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
    return best_w, best_iter, best_val

#### Stochastic Gradient Descent

In [84]:
def train_sgd(X, y, X_val, y_val, lr, schedule, batch_size=BATCH_SIZE):
    n, d = X.shape
    w = np.zeros(d)
    best_val, best_w, best_iter, wait = np.inf, w.copy(), 0, 0

    for k in range(MAX_ITER):
        eta = get_lr(lr, schedule, k)
        idx = np.random.randint(0, n, size=min(batch_size, n))
        Xb, yb = X[idx], y[idx]
        grad = 2.0 / len(idx) * Xb.T @ (Xb @ w - yb)

        with np.errstate(over='ignore', invalid='ignore'):
            w = w - eta * grad

        if (k + 1) % EVAL_EVERY == 0:
            vl = mse_loss(X_val, y_val, w)
            if vl < best_val:
                best_val, best_w, best_iter, wait = vl, w.copy(), k + 1, 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
    return best_w, best_iter, best_val

#### SAG (Stochastic Average Gradient)

In [85]:
def train_sag(X, y, X_val, y_val, lr, schedule):
    n, d = X.shape
    w = np.zeros(d)

    g = 2.0 * (X @ w - y)[:, None] * X
    avg_g = g.mean(axis=0)

    best_val, best_w, best_iter, wait = np.inf, w.copy(), 0, 0

    for k in range(MAX_ITER):
        eta = get_lr(lr, schedule, k)
        i = np.random.randint(0, n)

        new_gi = 2.0 * (X[i] @ w - y[i]) * X[i]
        avg_g = avg_g + (new_gi - g[i]) / n
        g[i] = new_gi

        with np.errstate(over='ignore', invalid='ignore'):
            w = w - eta * avg_g

        if (k + 1) % EVAL_EVERY == 0:
            vl = mse_loss(X_val, y_val, w)
            if vl < best_val:
                best_val, best_w, best_iter, wait = vl, w.copy(), k + 1, 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
    return best_w, best_iter, best_val

#### Momentum Descent

In [86]:
def train_momentum(X, y, X_val, y_val, lr, schedule, alpha=0.9):
    n, d = X.shape
    w = np.zeros(d)
    h = np.zeros(d)

    best_val, best_w, best_iter, wait = np.inf, w.copy(), 0, 0

    for k in range(MAX_ITER):
        eta = get_lr(lr, schedule, k)
        grad = full_grad(X, y, w)

        with np.errstate(over='ignore', invalid='ignore'):
            h = alpha * h + eta * grad
            w = w - h

        if (k + 1) % EVAL_EVERY == 0:
            vl = mse_loss(X_val, y_val, w)
            if vl < best_val:
                best_val, best_w, best_iter, wait = vl, w.copy(), k + 1, 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
    return best_w, best_iter, best_val

#### Adam

In [87]:
def train_adam(X, y, X_val, y_val, lr, schedule,
               beta1=0.9, beta2=0.999, eps=1e-8):
    n, d = X.shape
    w = np.zeros(d)
    m = np.zeros(d)
    v = np.zeros(d)

    best_val, best_w, best_iter, wait = np.inf, w.copy(), 0, 0

    for k in range(MAX_ITER):
        eta = get_lr(lr, schedule, k)
        grad = full_grad(X, y, w)

        m = beta1 * m + (1 - beta1) * grad
        v = beta2 * v + (1 - beta2) * (grad ** 2)

        t = k + 1
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        with np.errstate(over='ignore', invalid='ignore'):
            w = w - eta * m_hat / (np.sqrt(v_hat) + eps)

        if (k + 1) % EVAL_EVERY == 0:
            vl = mse_loss(X_val, y_val, w)
            if vl < best_val:
                best_val, best_w, best_iter, wait = vl, w.copy(), k + 1, 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    break
    return best_w, best_iter, best_val

#### Эксперименты

In [88]:
lr_grid = np.logspace(-5, 2, 15)
print('lr_grid:', lr_grid)

all_results = []

METHODS = {
    'VGD':      train_vgd,
    'SGD':      train_sgd,
    'SAG':      train_sag,
    'Momentum': train_momentum,
    'Adam':     train_adam,
}


def run_experiment(method, schedule, param_grid):
    history = []
    train_fn = METHODS[method]

    for p in param_grid:
        w, n_iter, val_loss_scaled = train_fn(
            X_train_b, y_train_s, X_val_b, y_val_s, p, schedule
        )
        m = final_metrics(w)
        m.update({'param': p, 'n_iter': n_iter,
                  'val_loss_scaled': val_loss_scaled})
        history.append(m)

    best = min(history, key=lambda h: h['val_loss_scaled'])

    return {
        'method': method,
        'schedule': schedule,
        'best_param': best['param'],
        'Loss_train': best['Loss_train'],
        'Loss_test':  best['Loss_test'],
        'R2_train':   best['R2_train'],
        'R2_test':    best['R2_test'],
        'iterations': best['n_iter'],
    }

lr_grid: [1.00000000e-05 3.16227766e-05 1.00000000e-04 3.16227766e-04
 1.00000000e-03 3.16227766e-03 1.00000000e-02 3.16227766e-02
 1.00000000e-01 3.16227766e-01 1.00000000e+00 3.16227766e+00
 1.00000000e+01 3.16227766e+01 1.00000000e+02]


4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [89]:
all_results.append(run_experiment('VGD', 'constant', lr_grid))
all_results[-1]

{'method': 'VGD',
 'schedule': 'constant',
 'best_param': np.float64(0.1),
 'Loss_train': 12769474.772987442,
 'Loss_test': 27594537.391286146,
 'R2_train': 0.7901911449097274,
 'R2_test': 0.6043083889874883,
 'iterations': 3950}

5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [90]:
all_results.append(run_experiment('VGD', 'time_decay', lr_grid))
all_results[-1]

{'method': 'VGD',
 'schedule': 'time_decay',
 'best_param': np.float64(1.0),
 'Loss_train': 13344538.296430402,
 'Loss_test': 26869800.641464576,
 'R2_train': 0.780742563695332,
 'R2_test': 0.6147007448378685,
 'iterations': 10000}

6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [91]:
all_results.append(run_experiment('SGD', 'constant', lr_grid))
all_results[-1]

{'method': 'SGD',
 'schedule': 'constant',
 'best_param': np.float64(0.03162277660168379),
 'Loss_train': 13740639.825735083,
 'Loss_test': 27007968.48976872,
 'R2_train': 0.7742344175232809,
 'R2_test': 0.6127194882684845,
 'iterations': 4600}

7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [92]:
all_results.append(run_experiment('SGD', 'time_decay', lr_grid))
all_results[-1]

{'method': 'SGD',
 'schedule': 'time_decay',
 'best_param': np.float64(0.31622776601683794),
 'Loss_train': 16213995.050688514,
 'Loss_test': 25000954.521592304,
 'R2_train': 0.7335959545320878,
 'R2_test': 0.6414990463067767,
 'iterations': 1850}

8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [93]:
all_results.append(run_experiment('SAG', 'constant', lr_grid))
all_results[-1]

{'method': 'SAG',
 'schedule': 'constant',
 'best_param': np.float64(0.00031622776601683794),
 'Loss_train': 21655529.03259352,
 'Loss_test': 28838915.918250993,
 'R2_train': 0.6441888305136892,
 'R2_test': 0.5864646347305467,
 'iterations': 1500}

9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [94]:
all_results.append(run_experiment('SAG', 'time_decay', lr_grid))
all_results[-1]

{'method': 'SAG',
 'schedule': 'time_decay',
 'best_param': np.float64(0.01),
 'Loss_train': 22015772.871438466,
 'Loss_test': 29257180.288850628,
 'R2_train': 0.638269844124267,
 'R2_test': 0.5804669366976032,
 'iterations': 400}

10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [95]:
all_results.append(run_experiment('Momentum', 'constant', lr_grid))
all_results[-1]

{'method': 'Momentum',
 'schedule': 'constant',
 'best_param': np.float64(0.31622776601683794),
 'Loss_train': 12939115.253427906,
 'Loss_test': 27450727.778103992,
 'R2_train': 0.7874038669980729,
 'R2_test': 0.6063705455916104,
 'iterations': 100}

11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [96]:
all_results.append(run_experiment('Momentum', 'time_decay', lr_grid))
all_results[-1]

{'method': 'Momentum',
 'schedule': 'time_decay',
 'best_param': np.float64(1.0),
 'Loss_train': 12875042.734380813,
 'Loss_test': 27486409.04152553,
 'R2_train': 0.7884566105214367,
 'R2_test': 0.6058588944482689,
 'iterations': 300}

12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [97]:
all_results.append(run_experiment('Adam', 'constant', lr_grid))
all_results[-1]

{'method': 'Adam',
 'schedule': 'constant',
 'best_param': np.float64(0.0001),
 'Loss_train': 14024490.596886829,
 'Loss_test': 26648818.504433405,
 'R2_train': 0.7695706074315909,
 'R2_test': 0.6178695161264376,
 'iterations': 5750}

13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [98]:
all_results.append(run_experiment('Adam', 'time_decay', lr_grid))
all_results[-1]

{'method': 'Adam',
 'schedule': 'time_decay',
 'best_param': np.float64(0.0031622776601683794),
 'Loss_train': 14319688.668597793,
 'Loss_test': 26512350.334497914,
 'R2_train': 0.7647203555182114,
 'R2_test': 0.6198264001737401,
 'iterations': 6000}

14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [99]:
df_res = pd.DataFrame(all_results)
df_res['method_name'] = df_res['method'] + ' (' + df_res['schedule'] + ')'

df_res['best_step'] = df_res.apply(
    lambda r: f"n = {r['best_param']:.2e}"
    if r['schedule'] == 'constant'
    else f"n(t) = {r['best_param']:.2e} * (1/(1+t))^0.5",
    axis=1
)

summary = df_res[
    ['method_name', 'best_step', 'Loss_train', 'Loss_test',
     'R2_train', 'R2_test', 'iterations']
].rename(columns={
    'method_name': 'Метод',
    'best_step':   'Лучший шаг',
    'iterations':  'Итераций'
})

summary

,Метод,Лучший шаг,Loss_train,Loss_test,R2_train,R2_test,Итераций
0,VGD (constant),n = 1.00e-01,1.276947e+07,2.759454e+07,0.790191,0.604308,3950
1,VGD (time_decay),n(t) = 1.00e+00 * (1/(1+t))^0.5,1.334454e+07,2.686980e+07,0.780743,0.614701,10000
2,SGD (constant),n = 3.16e-02,1.374064e+07,2.700797e+07,0.774234,0.612719,4600
3,SGD (time_decay),n(t) = 3.16e-01 * (1/(1+t))^0.5,1.621400e+07,2.500095e+07,0.733596,0.641499,1850
4,SAG (constant),n = 3.16e-04,2.165553e+07,2.883892e+07,0.644189,0.586465,1500
5,SAG (time_decay),n(t) = 1.00e-02 * (1/(1+t))^0.5,2.201577e+07,2.925718e+07,0.638270,0.580467,400
6,Momentum (constant),n = 3.16e-01,1.293912e+07,2.745073e+07,0.787404,0.606371,100
7,Momentum (time_decay),n(t) = 1.00e+00 * (1/(1+t))^0.5,1.287504e+07,2.748641e+07,0.788457,0.605859,300
8,Adam (constant),n = 1.00e-04,1.402449e+07,2.664882e+07,0.769571,0.617870,5750
9,Adam (time_decay),n(t) = 3.16e-03 * (1/(1+t))^0.5,1.431969e+07,2.651235e+07,0.764720,0.619826,6000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

#### 1) Лучший метод и шаг
**SGD (time_decay)**, шаг `n(t) = 0.316 · (1/(1+t))^0.5`. \
Минимальный `Loss_test = 2.50e+07`, максимальный `R2_test = 0.6415`. Разрыв `R2_train − R2_test = 0.092` — меньше, чем у `SGD (constant)` (0.161), то есть переобучение слабее.

Ближайший конкурент — `Adam (time_decay)` (`R2_test = 0.620`), но требует в 3 раза больше итераций.

#### 2) R²_train и R²_test простыми словами
- `R2_train` — доля разброса цены, объяснённая на обучающих данных.
- `R2_test` — доля разброса цены, объясняемая моделью на новых данных (обобщающая способность модели).

1 — идеально, 0 — не лучше среднего, < 0 — хуже среднего.

#### 3) Зачем оба
`R2_train` показывает подгонку под train, `R2_test` — обобщение. Сравнивать методы надо по `R2_test` и `Loss_test`, а `R2_train` нужен, чтобы увидеть переобучение: высокий train + низкий test = переобучение.